# Day 4 — LangChain and Toolchains
**GenAI 403 | Week 2**

### What we cover today
| # | Topic |
|---|-------|
| 1 | LLMChain & SimpleSequentialChain |
| 2 | Memory types (ConversationBuffer, Summary) |
| 3 | Retrievers (FAISS vector store) |
| 4 | Multi-chain RAG with memory |
| 5 | LangChain Agent with calculator + web search tools |
| 6 | Custom tool with @tool decorator |

> **Two tracks:**
> - 🔵 **Paid** — OpenAI API (`gpt-4o-mini` + `text-embedding-3-small`)
> - 🟢 **Free** — Ollama local LLM (`llama3.2`) + HuggingFace embeddings
>
> Set `USE_FREE = True` or `False` in Section 1 to switch between them.

---
## 0 — Install Dependencies
Run this cell once, then **restart the kernel** before continuing.

In [1]:
%pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-ollama \
    faiss-cpu \
    sentence-transformers \
    python-dotenv \
    duckduckgo-search \
    numexpr


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 1 — Load API Key & Choose Your Track

Create a file called `.env` in the **same folder as this notebook** containing:

```
OPENAI_API_KEY=sk-your-key-here
```

Get your key at: **platform.openai.com → API Keys → Create new secret key**

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env from the current directory

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("OpenAI key loaded successfully")
else:
    print("WARNING: OPENAI_API_KEY not found — check your .env file")

OpenAI key loaded successfully


In [3]:
# ── 🔵 PAID: OpenAI ────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

paid_llm = ChatOpenAI(
    model="gpt-4o-mini",    # swap to "gpt-4o" for more reasoning power
    temperature=0.3,
    api_key=OPENAI_API_KEY
)

paid_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

print("OpenAI LLM + Embeddings ready")

/Users/irumzahra/Library/CloudStorage/OneDrive-Personal/Desktop/GenAI403/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI LLM + Embeddings ready


In [ ]:
# ── 🟢 FREE: Ollama (local) + HuggingFace Embeddings ──────────────────────
# Setup steps (one-time):
#   1. Download and install Ollama from https://ollama.com
#   2. Open a terminal and run:  ollama pull llama3.2
#   3. Keep Ollama running in the background while using this notebook

from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

free_llm = ChatOllama(
    model="llama3.2",
    temperature=0.3
)

free_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # auto-downloads ~90MB on first run
)

print("Ollama LLM + HuggingFace Embeddings ready")

In [ ]:
# ── Choose your track — change this ONE line to switch for the whole notebook

USE_FREE = False   # False = OpenAI (paid) | True = Ollama + HuggingFace (free)

llm        = free_llm        if USE_FREE else paid_llm
embeddings = free_embeddings if USE_FREE else paid_embeddings

track = "🟢 FREE (Ollama + HuggingFace)" if USE_FREE else "🔵 PAID (OpenAI gpt-4o-mini)"
print("Active track:", track)

---
## 2 — LLMChain

**LLMChain** = a PromptTemplate + an LLM wired together.  
Variables inside `{}` in the template are filled in when you call `.invoke()`.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 1: define the prompt template
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in 3 bullet points. Be concise."
)

# Step 2: build the chain
chain = LLMChain(llm=llm, prompt=template)

# Step 3: run it
result = chain.invoke({"topic": "vector databases"})
print(result["text"])

---
## 3 — SimpleSequentialChain

The **output of Chain 1 automatically becomes the input of Chain 2**.  
Great for multi-step pipelines where each step builds on the previous one.

In [ ]:
from langchain.chains import SimpleSequentialChain

# Chain 1: summarize a concept in one paragraph
prompt_1 = PromptTemplate(
    input_variables=["concept"],
    template="Summarize the concept of '{concept}' in one short paragraph."
)
chain_1 = LLMChain(llm=llm, prompt=prompt_1)

# Chain 2: turn that summary into exam questions
# SimpleSequentialChain passes the previous output automatically
prompt_2 = PromptTemplate(
    input_variables=["summary"],
    template="Based on this explanation:\n{summary}\n\nWrite 2 short exam questions."
)
chain_2 = LLMChain(llm=llm, prompt=prompt_2)

# Wire them: chain_1 output → chain_2 input
seq_chain = SimpleSequentialChain(
    chains=[chain_1, chain_2],
    verbose=True   # prints what each chain produces
)

result = seq_chain.invoke("Retrieval-Augmented Generation")
print("\n--- Final Output ---")
print(result["output"])

---
## 4 — Memory Types

Memory lets a chain remember previous conversation turns.

| Type | Stores | Best for |
|---|---|---|
| `ConversationBufferMemory` | Every message verbatim | Short conversations |
| `ConversationSummaryMemory` | A compressed LLM summary | Long conversations (saves tokens) |

In [ ]:
# ── 4a: Buffer Memory — stores the full chat history verbatim ─────────────
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

buffer_memory = ConversationBufferMemory()
convo_chain = ConversationChain(llm=llm, memory=buffer_memory, verbose=False)

turns = [
    "My name is Ali.",
    "What is the capital of France?",
    "What is my name?"    # tests if the model remembers turn 1
]

print("=== Buffer Memory ===")
for turn in turns:
    print(f"User : {turn}")
    response = convo_chain.invoke(turn)
    print(f"Bot  : {response['response']}\n")

In [ ]:
# ── 4b: Summary Memory — LLM compresses history into a running summary ─────
from langchain.memory import ConversationSummaryMemory

summary_memory = ConversationSummaryMemory(llm=llm)
summary_chain = ConversationChain(llm=llm, memory=summary_memory, verbose=False)

turns = [
    "I am building a chatbot for a hospital.",
    "It should answer patient queries about appointments.",
    "What should I focus on to make it reliable?"
]

print("=== Summary Memory ===")
for turn in turns:
    print(f"User : {turn}")
    response = summary_chain.invoke(turn)
    print(f"Bot  : {response['response']}\n")

print("--- Compressed Memory Buffer ---")
print(summary_memory.buffer)

---
## 5 — Retriever with FAISS Vector Store

A **retriever** finds the most relevant document chunks for a query using vector similarity.  
FAISS stores everything in memory — no external database or API needed.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

# Sample documents — in a real project these would come from files/databases
raw_docs = [
    "LangChain is a framework for building applications powered by language models. It supports chains, agents, and memory.",
    "FAISS (Facebook AI Similarity Search) is a library for fast vector similarity search. It stores embeddings and retrieves nearest neighbors.",
    "RAG (Retrieval-Augmented Generation) combines a retriever and a generator. It grounds LLM responses in real documents, reducing hallucinations.",
    "Memory in LangChain allows conversational chains to track context across turns. Types include buffer memory, summary memory, and entity memory.",
    "Agents use tools to take actions. A LangChain agent can call a calculator, search the web, or query a database based on the user request."
]

# Wrap strings as Document objects
docs = [Document(page_content=text) for text in raw_docs]

# Split into chunks (these are short, so each stays as one chunk)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

# Build FAISS vector store: embed each chunk and store
vector_store = FAISS.from_documents(chunks, embeddings)

# Create a retriever — k=2 means return top 2 most similar chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Test retrieval
query = "How does RAG reduce hallucinations?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}: {doc.page_content}")

---
## 6 — Multi-chain RAG with Memory

`ConversationalRetrievalChain` combines three things:
1. **Retriever** → fetches relevant chunks from the vector store
2. **Memory** → tracks the full conversation history
3. **LLM** → generates an answer grounded in retrieved context

This is the core pattern of a production RAG chatbot.

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# memory_key must be 'chat_history' for ConversationalRetrievalChain
rag_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=rag_memory,
    verbose=True
)

# Multi-turn conversation
# Notice Q2 uses 'it' to refer to FAISS from Q1 — memory resolves the reference
questions = [
    "What is FAISS?",
    "How does it relate to RAG?",
    "What memory types does LangChain support?"
]

for question in questions:
    print(f"\nQ: {question}")
    answer = rag_chain.invoke({"question": question})
    print(f"A: {answer['answer']}")

---
## 7 — LangChain Agent with Tools

An **agent** reads the user query and decides which tool to call.  
It follows a Reason → Act loop until it has a final answer.

| Tool | What it does | Cost |
|---|---|---|
| `llm-math` | Evaluates math expressions | Free (runs locally) |
| `ddg-search` | Web search via DuckDuckGo | Free (no API key) |

In [ ]:
from langchain.agents import initialize_agent, load_tools, AgentType

# Load built-in tools
tools = load_tools(
    ["llm-math", "ddg-search"],
    llm=llm    # llm-math needs the LLM to parse the math expression
)

# Initialize agent with ZERO_SHOT_REACT_DESCRIPTION
# This agent reasons about which tool to use based on tool descriptions alone
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True    # shows Thought / Action / Observation steps
)

print("Agent ready. Tools:", [t.name for t in tools])

In [ ]:
# Test 1: Math — agent should pick llm-math
result = agent.invoke("What is 17 raised to the power of 3, divided by 4.9?")
print("\nFinal Answer:", result["output"])

In [ ]:
# Test 2: Web search — agent should pick ddg-search
result = agent.invoke("Who founded LangChain and when was it first released?")
print("\nFinal Answer:", result["output"])

In [ ]:
# Test 3: Multi-step — agent chains search → math
result = agent.invoke(
    "Search for the current population of Pakistan, then calculate 10% of that number."
)
print("\nFinal Answer:", result["output"])

---
## 8 — Custom Tool with @tool

Any Python function can be turned into a LangChain tool with `@tool`.  
The **docstring** is critical — the agent reads it to decide when to use the tool.

In [ ]:
from langchain.tools import tool
from langchain.agents import initialize_agent, AgentType

@tool
def word_count(text: str) -> str:
    """Counts the number of words in the given text. Input should be a plain string."""
    count = len(text.split())
    return f"The text has {count} words."

@tool
def reverse_text(text: str) -> str:
    """Reverses the characters in the given text string. Input should be a plain string."""
    return f"Reversed: {text[::-1]}"

# Build agent with only our custom tools
custom_agent = initialize_agent(
    tools=[word_count, reverse_text],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Test 1: word count
result = custom_agent.invoke(
    "How many words are in: 'LangChain makes building LLM apps easy'?"
)
print("\nFinal Answer:", result["output"])

In [ ]:
# Test 2: reverse text
result = custom_agent.invoke("Reverse the text: 'hello world'")
print("\nFinal Answer:", result["output"])

---
## Summary

| Concept | Key Class / Function |
|---|---|
| **LLMChain** | `PromptTemplate` + `LLMChain` |
| **Sequential Chain** | `SimpleSequentialChain` |
| **Buffer Memory** | `ConversationBufferMemory` — stores full history |
| **Summary Memory** | `ConversationSummaryMemory` — compresses history |
| **Retriever** | `FAISS.from_documents()` + `.as_retriever()` |
| **RAG + Memory** | `ConversationalRetrievalChain` |
| **Agent + Tools** | `initialize_agent` with `llm-math`, `ddg-search` |
| **Custom Tool** | `@tool` decorator on any Python function |

### Free vs Paid
| Component | 🔵 Paid | 🟢 Free |
|---|---|---|
| LLM | OpenAI `gpt-4o-mini` | Ollama `llama3.2` (runs locally) |
| Embeddings | OpenAI `text-embedding-3-small` | HuggingFace `all-MiniLM-L6-v2` |
| Web Search | DuckDuckGo (free in both tracks) | DuckDuckGo (free in both tracks) |
| Vector Store | FAISS (free in both tracks) | FAISS (free in both tracks) |

> **Next session:** Day 5 — LLM Evaluation, Guardrails & Hallucination Testing